In [35]:
library(forecast)
library(M4comp2018)
library(dplyr)
library(M4metalearning)

In [2]:
nnetarl=load('Weekly_FForma_datatestlist.RData')
nnetar_datalist <- eval(parse(text = nnetarl))

In [3]:
FForma_predh=nnetar_datalist[[2]]

In [5]:
FForma_pred_res=nnetar_datalist[[1]]

In [7]:
MASE=FForma_pred_res[,,,6]
m=5

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [11]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
 0  1  2  3  4 
16  9 14 38 37 

In [13]:
FFormaoptl=load('Weekly_FForma_opt_pre_result.RData')
FForma_opt_pre_list<- eval(parse(text = nnetaroptl))

In [14]:
time_matrix <- matrix(0,ncol = 3, nrow =3)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [15]:
set.seed(100)
index = sample(3,dim(FForma_opt_pre_list[[1]]),replace = TRUE,prob=c(0.3,0.4,0.3))

In [16]:
FForma_pr_min=FForma_opt_pre_list[[1]][index==3,]
FForma_pr_mean=FForma_opt_pre_list[[2]][index==3,]

In [18]:
FForma_pr_min=FForma_pr_min+1
FForma_pr_mean=FForma_pr_mean+1

In [20]:
round5=function(x)
    {
    l=round(x)
    for(i in 1:length(l))
    {
    if(l[i]>5)
        {
        l[i]=5
    }
    if(l[i]<1)
    {
        l[i]=1
    }
}
    return(l)
}

In [21]:
FForma_pr_min[,2]=round5(FForma_pr_min[,2])
FForma_pr_min[,4]=round5(FForma_pr_min[,4])

In [22]:
FForma_pr_mean[,2]=round5(FForma_pr_mean[,2])
FForma_pr_mean[,4]=round5(FForma_pr_mean[,4])

In [23]:
realbestmin

4
0
2
0
4
4
0
2
3
0
0


In [24]:
head(FForma_pr_min)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
4,3,2,4
5,3,5,2
5,2,5,4
2,2,2,2
4,3,3,4
5,5,4,5


In [25]:
Weekly_M4 <- Filter(function(l) l$period == "Weekly", M4)

In [26]:
data=Weekly_M4

In [27]:
datalength=length(data)
h=data[[1]]$h
m=5
n=4
freq=frequency(data[[1]]$x)

In [28]:
realxx=matrix(0,datalength,h)
for(i in 1:datalength){
    realxx[i,]=data[[i]]$xx
}

In [29]:
length5=matrix(0,datalength,m)
for(k in 1:datalength)
{
y = data[[k]]$x
y_l=length(y)
loc = 1:length(y)
loc_m = as.integer(loc*m/length(y))
filt_d = data.frame(y,loc_m)
filt_0=filter(filt_d ,loc_m==0)
m_l=count(filt_0)
m_l=as.integer(m_l)
length5[k,]=c(0,1*m_l,2*m_l,3*m_l,4*m_l)
    }

In [30]:
length5=length5[index==3,]

In [31]:
end_time = Sys.time()
time_matrix[1,]=end_time-start_time

In [32]:
end_time-start_time

Time difference of 24.94451 secs

In [36]:
set.seed(100)
index = sample(3,length(data),replace = TRUE,prob=c(0.3,0.4,0.3))
tindex=c(1:length(data))[index==3]
M4_test <- data[tindex]
M4_test <- temp_holdout(M4_test)

## FForma

In [37]:
start_time = Sys.time()

In [38]:
# set.seed(100)
# index = sample(3,length(data),replace = TRUE,prob=c(0.3,0.4,0.3))
testindex=c(1:dim(nnetar_pr_min)[1])

In [40]:
meanpre=function(pred_array,opt_pr,h,testindex)
    {
    data_length=length(testindex)
    meanpreh=matrix(0,data_length,h)
    for (i in 1:data_length){
    index=testindex[i]
    loc=opt_pr[index]
    #print(index,loc)
    predh=pred_array[index,loc,,]
    predh=unique(predh)
    meanpreh[i,]=apply(predh,2,mean)
}
    return(meanpreh)
}

In [41]:
predres=function(pred_matrix,data,opt_pr,lengthmatrix,h,testindex)
    {
    data_length=length(testindex)
    pred_res=matrix(0,data_length,6)
    model=ets(data[[1]]$x)
    fore_l=forecast(model,h=h)
    freq=frequency(data[[1]]$x)
    for (i in 1:data_length){
    index=testindex[i]
    loc=opt_pr[index]
    start=lengthmatrix[i,loc]
    y_all=data[[index]]$x
    y=y_all[start:length(y_all)]
    fore_l$x=ts(y,frequency=freq,end=end(y_all))
    real=data[[index]]$xx
    fore_l$mean=ts(pred_matrix[i,],start=start(real),frequency=freq)
    res=accuracy(fore_l,real)
    pred_res[i,]=res[2,1:6]
    }
    return(pred_res)
    }

In [42]:
predres_mul=function(pred_matrix,data,opt_pr_mul,lengthmatrix,h,testindex)
    {
    data_length=length(testindex)
    pred_res=matrix(0,data_length,6)
    model=ets(data[[1]]$x)
    fore_l=forecast(model,h=h)
    freq=frequency(data[[1]]$x)
    for (i in 1:data_length){
    index=testindex[i]
    loc=min(opt_pr_mul[index,])
    start=lengthmatrix[i,loc]
    y_all=data[[index]]$x
    y=y_all[start:length(y_all)]
    fore_l$x=ts(y,frequency=freq,end=end(y_all))
    real=data[[index]]$xx
    fore_l$mean=ts(pred_matrix[i,],start=start(real),frequency=freq)
    res=accuracy(fore_l,real)
    pred_res[i,]=res[2,1:6]
    }
    return(pred_res)
    }

In [43]:
FFormaxgbcls_min=meanpre(FForma_predh,FForma_pr_min[,1],h,testindex)
FFormaxgbreg_min=meanpre(FForma_predh,FForma_pr_min[,2],h,testindex)
FFormalgbcls_min=meanpre(FForma_predh,FForma_pr_min[,3],h,testindex)
FFormalgbreg_min=meanpre(FForma_predh,FForma_pr_min[,4],h,testindex)

In [44]:
FFormaxgbcls_mean=meanpre(FForma_predh,FForma_pr_mean[,1],h,testindex)
FFormaxgbreg_mean=meanpre(FForma_predh,FForma_pr_mean[,2],h,testindex)
FFormalgbcls_mean=meanpre(FForma_predh,FForma_pr_mean[,3],h,testindex)
FFormalgbreg_mean=meanpre(FForma_predh,FForma_pr_mean[,4],h,testindex)

### The error of each improved prediction result

In [46]:
lengthtest=length5[testindex,]

In [47]:
FFormaminxgbclsres=predres(FFormaxgbcls_min,M4_test,FForma_pr_min[,1],lengthtest,h,testindex)
FFormaminxgbregres=predres(FFormaxgbreg_min,M4_test,FForma_pr_min[,2],lengthtest,h,testindex)
FFormaminlgbclsres=predres(FFormalgbcls_min,M4_test,FForma_pr_min[,3],lengthtest,h,testindex)
FFormaminlgbregres=predres(FFormalgbreg_min,M4_test,FForma_pr_min[,4],lengthtest,h,testindex)

In [48]:
FFormameanxgbclsres=predres(FFormaxgbcls_mean,M4_test,FForma_pr_mean[,1],lengthtest,h,testindex)
FFormameanxgbregres=predres(FFormaxgbreg_mean,M4_test,FForma_pr_mean[,2],lengthtest,h,testindex)
FFormameanlgbclsres=predres(FFormalgbcls_mean,M4_test,FForma_pr_mean[,3],lengthtest,h,testindex)
FFormameanlgbregres=predres(FFormalgbreg_mean,M4_test,FForma_pr_mean[,4],lengthtest,h,testindex)

In [91]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

In [92]:
end_time-start_time

Time difference of 14.37467 mins

### The results of the two methods of determining labels are averaged respectively

In [61]:
FFormaallmin=(FFormaxgbcls_min+FFormalgbcls_min+FFormaxgbreg_min+FFormalgbreg_min)/4
FFormaallmean=(FFormaxgbcls_mean+FFormalgbcls_mean+FFormaxgbreg_mean+FFormalgbreg_mean)/4

In [62]:
FFormaallminres=predres_mul(FFormaallmin,M4_test,FForma_pr_min,lengthtest,h,testindex)
FFormaallmeanres=predres_mul(FFormaallmean,M4_test,FForma_pr_mean,lengthtest,h,testindex)

### Under the above classification, the improved forecast results of the classification and regression models are averaged respectively

In [59]:
FFormaclsmin=(FFormaxgbcls_min+FFormalgbcls_min)/2
FFormaregmin=(FFormaxgbreg_min+FFormalgbreg_min)/2
FFormaclsmean=(FFormaxgbcls_mean+FFormalgbcls_mean)/2
FFormaregmean=(FFormaxgbreg_mean+FFormalgbreg_mean)/2

In [60]:
FFormaclsminres=predres_mul(FFormaclsmin,M4_test,FForma_pr_min[,c(1,3)],lengthtest,h,testindex)
FFormaregminres=predres_mul(FFormaregmin,M4_test,FForma_pr_min[,c(2,4)],lengthtest,h,testindex)
FFormaclsmeanres=predres_mul(FFormaclsmean,M4_test,FForma_pr_mean[,c(1,3)],lengthtest,h,testindex)
FFormaregmeanres=predres_mul(FFormaregmean,M4_test,FForma_pr_mean[,c(2,4)],lengthtest,h,testindex)

### Random slice

In [1]:
# FFormapred_change<- aperm(FForma_predh[1:datalength,1:m,1:n,1:h], c(1, 3, 2, 4))  
# # 使用array函数将其转换为新的形状
# FFormapred_change <- array(FFormapred_change, dim = c(datalength, m*n, h))

In [82]:
# data$x

NULL

In [40]:
# model <- FForma(data[[1]]$x)
# fore_l <- forecast(model, h = h)
# FFormarandomres <- matrix(0, datalength, 6)
# for (i in 1:datalength) {
#     random <- sample(m * n, n)
#     minloc <- min(random)
#     # 使用cut函数将数据分割为区间
#     loc <- cut(minloc, breaks = seq(0, m * n, n), labels = FALSE)
#     loc <- as.numeric(loc)
#     FFormarandompred <- apply(FFormapred_change[i, random, ], 2, mean)
#     start <- length5[i, loc]
#     y_all <- data[[i]]$x
#     y <- y_all[start:length(y_all)]
#     fore_l$x <- ts(y, frequency = freq, end = end(y_all))
#     real <- data[[i]]$xx
#     fore_l$mean <- ts(FFormarandompred, start = start(real), frequency = freq)
#     res <- accuracy(fore_l, real)
#     FFormarandomres[i, ] <- res[2, 1:6]
# }

Warning message in .cbind.ts(list(e1, e2), c(deparse(substitute(e1))[1L], deparse(substitute(e2))[1L]), :
“non-intersecting series”
Warning message in trainingaccuracy(object, test, d, D):
“test elements must be within sample”


In [52]:
mean1=function(x)
    {
# 使用is.finite函数检查哪些值是有限的
finite_values <- x[is.finite(x)]
# 计算有限值的平均值
d=mean(finite_values, na.rm = TRUE)
return(d)
}

In [53]:
print(apply(nnetar_pred_res[,4,4,],2,mean1))

[1]   1.575504 406.822895 341.189622  -3.559800  10.763751   2.138815


In [54]:
max(nnetarmeanxgbclsres[,6])

[1] 9.78144

In [55]:
max(nnetar_pred_res[,1,1,6])

[1] 10.10853

In [56]:
print(apply(FForma_pred_res[,1,1,],2,mean1))
print(apply(FFormaminxgbclsres,2,mean1))
print(apply(FFormaminxgbregres,2,mean1))
print(apply(FFormaminlgbclsres,2,mean1))
print(apply(FFormaminlgbregres,2,mean1))

[1] -38.836800 396.455901 336.202963  -4.771765   9.415614   1.999592
[1] -24.354229 395.730054 328.536291  -3.991557   9.821117   1.657883
[1] -29.569874 396.654839 329.645796  -4.391414  10.301632   1.642057
[1] -38.110967 384.895052 319.575160  -5.017623  10.072669   1.998593
[1] -40.444780 391.389919 325.567073  -4.835228  10.475662   1.721188


In [57]:
print(apply(FFormameanxgbclsres,2,mean1))
print(apply(FFormameanxgbregres,2,mean1))
print(apply(FFormameanlgbclsres,2,mean1))
print(apply(FFormameanlgbregres,2,mean1))

[1] -32.415743 396.721038 333.389033  -4.762993   9.645815   1.624219
[1] -50.489320 389.507646 325.132233  -5.009860  10.075844   1.608162
[1] -42.868995 394.266788 332.987518  -5.013390   9.954198   1.952383
[1] -58.625440 381.118806 318.772419  -5.261852  10.014482   1.674664


In [64]:
print(apply(FFormaallminres,2,mean1))
print(apply(FFormaallmeanres,2,mean1))

[1] -33.119963 387.521160 321.866382  -4.558956   9.998645   1.822820
[1] -46.099875 384.586842 322.553396  -5.012023   9.757696   1.807222


In [63]:
print(apply(dd,2,mean1))

[1] 238.420504 614.919147 544.642072   1.973166  14.057852   3.985586


In [65]:
print(apply(FFormaclsminres,2,mean1))
print(apply(FFormaregminres,2,mean1))
print(apply(FFormaclsmeanres,2,mean1))
print(apply(FFormaregmeanres,2,mean1))

[1] -31.232598 386.529156 321.057277  -4.504590   9.826198   1.845347
[1] -35.007327 392.410177 326.080279  -4.613321  10.335826   1.684428
[1] -37.642369 390.847701 329.539995  -4.888191   9.688176   1.824465
[1] -54.557380 383.267883 320.230950  -5.135856   9.981218   1.640127


In [46]:
print(apply(FFormarandomres[testindex,],2,mean))

[1] 127.8319874 448.8477151 370.3941851   0.6034996   8.6255497   3.1012827


In [66]:
FFormaall <- matrix(0, 16, 6)
FFormaall[1, ] <- apply(FForma_pred_res[testindex, 1, 1, ], 2, mean1)[1:6]
FFormaall[2, ] <- apply(FFormaminxgbclsres, 2, mean1)
FFormaall[3, ] <- apply(FFormaminxgbregres, 2, mean1)
FFormaall[4, ] <- apply(FFormaminlgbclsres, 2, mean1)
FFormaall[5, ] <- apply(FFormaminlgbregres, 2, mean1)
FFormaall[6, ] <- apply(FFormameanxgbclsres, 2, mean1)
FFormaall[7, ] <- apply(FFormameanxgbregres, 2, mean1)
FFormaall[8, ] <- apply(FFormameanlgbclsres, 2, mean1)
FFormaall[9, ] <- apply(FFormameanlgbregres, 2, mean1)
FFormaall[10, ] <- apply(FFormaallminres, 2, mean1)
FFormaall[11, ] <- apply(FFormaallmeanres, 2, mean1)
FFormaall[12, ] <- apply(FFormaclsminres, 2, mean1)
FFormaall[13, ] <- apply(FFormaregminres, 2, mean1)
FFormaall[14, ] <- apply(FFormaclsmeanres, 2, mean1)
FFormaall[15, ] <- apply(FFormaregmeanres, 2, mean1)
# FFormaall[16, ] <- apply(FFormarandomres[testindex, ], 2, mean)

In [49]:
FFormaall

130.54199,512.2310,425.6161,-0.07025813,9.914318,3.858539
102.76203,441.1372,364.3545,0.47872087,8.704624,2.274993
92.03216,432.0178,361.2093,0.57789827,8.714427,2.213900
102.34957,458.7778,379.8591,0.28377300,8.825521,2.927185
66.49002,428.7702,360.1808,-0.14712318,8.718876,2.169807
95.61926,452.3280,371.3381,0.57183024,8.885383,2.313615
108.41586,441.3150,364.1451,0.39070912,8.549537,2.612842
154.32508,476.6280,394.1587,0.94372699,8.943917,2.885883
88.80144,442.0213,364.4407,0.24903784,8.779797,2.273817
90.90845,420.0109,350.7146,0.29831724,8.260041,2.402083
111.79041,437.7479,362.3980,0.53882605,8.499255,2.556476


In [67]:
write.csv(FFormaall,'FForma_Weekly_final_res.csv')